In [1]:
# models，prompt和Parsers
# 与课程略有区别的是，openai包1.0.0版本过后，所有的请求都需要挂在client对象上
# 如果希望和课程的代码格式保持一致需要下载版本更低的langchain
# 导包
import os
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv()) # read local .env file
# 获得DEEPSEEK_API_KEY
client = OpenAI(
    api_key = os.environ.get("DEEPSEEK_API_KEY"),
    base_url = "https://api.deepseek.com"
)


In [2]:

def get_completion(prompt, model = "deepseek-v4-flash"):
    message = [
        {
            "role":"user",
            "content":prompt
        }
    ]
    response = client.chat.completions.create(
        model = model,
        messages = message,
        temperature = 0,
    )
    return response.choices[0].message.content

In [3]:
get_completion("What is the answer of 1+1")

'1+1 = 2'

In [4]:
customer_email = """
Arr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [5]:
style = """American English in a calm and respectful tone""" # 转换风格

In [6]:
prompt = f"""Translate the text \
              that is delimit by triple backticks into a style that is {style},
              text : ```{customer_email}```
""" # 格式化字面量

print(prompt)

Translate the text               that is delimit by triple backticks into a style that is American English in a calm and respectful tone,
              text : ```
Arr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



In [7]:
response = get_completion(prompt)

In [8]:
response

'I’m frustrated that my blender lid came off and splattered smoothie all over my kitchen walls. To make things worse, the warranty doesn’t cover the cost of cleaning up. I could really use your help with this.'

In [9]:
# use with langChain，新版需要安装新的langChain包
# !pip install langchain-openai
# langChain 能够减轻上述步骤的复杂度
from langchain_openai import ChatOpenAI

In [10]:
# 由于我此处调用的DEEPSEEK模型，因此还需要传入部分参数
chat = ChatOpenAI(
    api_key = os.environ.get("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
    model="deepseek-v4-flash",
    temperature=0.0) # temperature = 0.0能够降低输出的随机性
chat

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x00000256D01A57C0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000256CEEEAA50>, root_client=<openai.OpenAI object at 0x00000256D01DBF20>, root_async_client=<openai.AsyncOpenAI object at 0x00000256CF018140>, model_name='deepseek-v4-flash', temperature=0.0, openai_api_key=SecretStr('**********'), openai_api_base='https://api.deepseek.com', openai_proxy='')

In [11]:
# 创建字符串模板
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

In [12]:
# 为了能够重复使用上述字符串模板，需要导入新的软件包
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(template_string)
prompt_template.messages[0].prompt.input_variables

['style', 'text']

In [13]:
# 指定风格
customer_style = """Chinese in a calm and respectful tone"""

In [14]:
customer_email = """
Arr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [15]:
customer_messages = prompt_template.format_messages(
    style=customer_style,
    text=customer_email
)
customer_messages

[HumanMessage(content="Translate the text that is delimited by triple backticks into a style that is Chinese in a calm and respectful tone. text: ```\nArr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!\n```\n")]

In [16]:
print(type(customer_messages))
print(type(customer_messages[0]))

<class 'list'>
<class 'langchain_core.messages.human.HumanMessage'>


In [17]:
print(customer_messages[0])

content="Translate the text that is delimited by triple backticks into a style that is Chinese in a calm and respectful tone. text: ```\nArr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!\n```\n"


In [18]:
customer_response = chat.invoke(customer_messages) # 新版langchain调用模型需要在chat后加入invoke

In [19]:
print(customer_response.content)

我感到十分无奈和恼火，因为我的搅拌机盖子突然飞脱，把思慕雪溅得到处都是，厨房墙壁也没能幸免。更糟糕的是，保修条款并不包含清理厨房的费用。我现在非常需要您的帮助，拜托您了。


In [20]:
"""
使用 LangChain 调用大模型的步骤：
1. 创建模型对象：从 langchain_openai 导入 ChatOpenAI，
   指定 model、api_key、base_url，声明 chat 实体
2. 创建模板对象：写好含 {变量} 占位符的模板字符串，
   通过 langchain_core.prompts 中的 ChatPromptTemplate.from_template()
   将其转换为模板对象
3. 填充变量生成消息列表：调用模板对象的 format_messages 方法，
   传入各变量的实际值，得到消息列表
4. 调用模型：对 chat 实体调用 invoke 方法并传入消息列表，
   得到响应对象，通过 .content 取出回答内容
"""

'\n使用 LangChain 调用大模型的步骤：\n1. 创建模型对象：从 langchain_openai 导入 ChatOpenAI，\n   指定 model、api_key、base_url，声明 chat 实体\n2. 创建模板对象：写好含 {变量} 占位符的模板字符串，\n   通过 langchain_core.prompts 中的 ChatPromptTemplate.from_template()\n   将其转换为模板对象\n3. 填充变量生成消息列表：调用模板对象的 format_messages 方法，\n   传入各变量的实际值，得到消息列表\n4. 调用模型：对 chat 实体调用 invoke 方法并传入消息列表，\n   得到响应对象，通过 .content 取出回答内容\n'

In [21]:
# exercise 2 ：客服代表用客户的原始语言回复客户
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

In [22]:
service_style_private = """\
a polite tone \
that speaks in English Pirate
"""

In [23]:
# 使用模板
service_message = prompt_template.format_messages(
    style=service_style_private,
    text=service_reply
)

print(service_message[0].content)

Translate the text that is delimited by triple backticks into a style that is a polite tone that speaks in English Pirate
. text: ```Hey there customer, the warranty does not cover cleaning expenses for your kitchen because it's your fault that you misused your blender by forgetting to put the lid on before starting the blender. Tough luck! See ya!
```



In [24]:
# 调用模型
service_response = chat.invoke(service_message)
service_response.content

'Ahoy there, valued customer. We be regretful to inform ye that the warranty cannot cover the cost o’ swabbin’ yer galley, for the blender was set awhirl without its lid — a misuse, alas, under our policy. We thank ye fer yer understandin’, and may fair winds fill yer sails!'

In [25]:
# 为什么使用提示模板而非"f"字符串?
"""
当构建较为复杂的应用程序时，Prompt可能会很长且详细，
prompt template能够更方便的进行重用

同样的，LangChain为一些常见的操作提供了提示，例如摘要、回答问题、
连接到SQL数据库或者连接到不同的API
通过使用LangChain的内置提示，可以快速的让程序运行，无需设计自己的提示

LangChain的提示库还支持输出解析，即使用LLM构建一个复杂的应用程序时，通常会指示LLM以某种格式生成输出
"""

'\n当构建较为复杂的应用程序时，Prompt可能会很长且详细，\nprompt template能够更方便的进行重用\n\n同样的，LangChain为一些常见的操作提供了提示，例如摘要、回答问题、\n连接到SQL数据库或者连接到不同的API\n通过使用LangChain的内置提示，可以快速的让程序运行，无需设计自己的提示\n\nLangChain的提示库还支持输出解析，即使用LLM构建一个复杂的应用程序时，通常会指示LLM以某种格式生成输出\n'

In [26]:
# LangChain的输出解释器：获取JSON输出格式的数据
# 输出格式示例
{
    "gift" : False,
    "delivery_days" : 5,
    "price_value" : "pretty affordable!"
}

{'gift': False, 'delivery_days': 5, 'price_value': 'pretty affordable!'}

In [27]:
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn.\
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

# 评论模板要求LLM将客户模板作为输入，提取这三个字段，将格式输出为JSON

review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.
delivery_days: How many days did it take for the product to arrive? If this information is not found, output -1.
price_value: Extract any sentences about the value or price,and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

In [28]:
# 创建一个新的提示词模板
prompt_template = ChatPromptTemplate.from_template(review_template)
"""
prompt_template                # ChatPromptTemplate 对象
    .messages                  # 模板里的消息模板列表（list[MessagePromptTemplate]）
    [0]                        # 取第一条消息模板（通常是 HumanMessagePromptTemplate）
    .prompt                    # 该消息模板内部封装的 PromptTemplate 对象
    .input_variables           # 该 PromptTemplate 里出现的 {变量} 名列表（list[str]）
"""
prompt_template.messages[0].prompt.input_variables

['text']

In [29]:
message = prompt_template.format_messages(text=customer_review)
response = chat.invoke(message)
response.content

'{\n  "gift": true,\n  "delivery_days": 2,\n  "price_value": ["It\'s slightly more expensive than the other leaf blowers out there, but I think it\'s worth it for the extra features."]\n}'

In [54]:
type(response.content) # json实际上为一个字符串

str

In [31]:

# 使用LangChain解析字符串数据
# 新版LangChain中，ResponseSchema和StructuredOutputParser已移至langchain_classic包
from langchain_classic.output_parsers import ResponseSchema
from langchain_classic.output_parsers import StructuredOutputParser


In [32]:
# 需要解析的东西
gift_schema = ResponseSchema(name="gift", description="Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.")
delivery_days_schema = ResponseSchema(name="delivery_days", description="How many days did it take for the product to arrive?")
price_value_schema = ResponseSchema(name="price_value", description="Extract any sentences about the value or price")

response_schema = [gift_schema, delivery_days_schema, price_value_schema]

In [33]:
output_parsers = StructuredOutputParser.from_response_schemas(response_schema)

In [34]:
format_instructions = output_parsers.get_format_instructions()
format_instructions

'The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":\n\n```json\n{\n\t"gift": string  // Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\t"delivery_days": string  // How many days did it take for the product to arrive?\n\t"price_value": string  // Extract any sentences about the value or price\n}\n```'

In [36]:
review_template_2 = """
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? Answer \
with 'yes' or 'no'.
delivery_days: How many days did it take for the product to arrive?\
 If this information is not found, output 'no information found.'
price_value: Extract any sentences about the value or price,\
 and output them as a comma separated Python list.
text: {text}
{format_instructions}
"""

prompt = ChatPromptTemplate.from_template(template=review_template_2)

messages = prompt.format_messages(
    text=customer_review,
    format_instructions=format_instructions

)



In [39]:
print(messages[0].content)


For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? Answer with 'yes' or 'no'.
delivery_days: How many days did it take for the product to arrive? If this information is not found, output 'no information found.'
price_value: Extract any sentences about the value or price, and output them as a comma separated Python list.
text: This leaf blower is pretty amazing.  It has four settings:candle blower, gentle breeze, windy city, and tornado. It arrived in two days, just in time for my wife's anniversary present. I think my wife liked it so much she was speechless. So far I've been the only one using it, and I've been using it every other morning to clear the leaves on our lawn.It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features.

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```

In [41]:
# 调用DEEPSEEK API获取返回结果
response = chat.invoke(messages) # chat方案被废弃，但仍然可以使用
response.content

'```json\n{\n\t"gift": "yes",\n\t"delivery_days": "2",\n\t"price_value": "It\'s slightly more expensive than the other leaf blowers out there, but I think it\'s worth it for the extra features."\n}\n```'

In [44]:
print(response.content)

```json
{
	"gift": "yes",
	"delivery_days": "2",
	"price_value": "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."
}
```


In [46]:
output_dict = output_parsers.parse(response.content)

In [56]:
type(output_dict)

dict

In [60]:
output_dict.get('gift') # 可以直接根据key取值
output_dict.get('delivery_days') # 可以直接根据key取值

'2'